# Motion detection notebook for CCTV-style clips



In [ ]:
from pathlib import Path
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "requirements.txt").exists() and not (ROOT / ".git").exists():
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"Project root: {ROOT}")
print(f"Source path: {SRC}")

In [ ]:
def detect_motion_frames(video_path: str, max_frames: int = 150, min_area: int = 900):
    capture = cv2.VideoCapture(str(video_path))
    subtractor = cv2.createBackgroundSubtractorMOG2(history=300, varThreshold=25, detectShadows=True)
    results = []

    frame_count = 0
    while capture.isOpened() and frame_count < max_frames:
        success, frame = capture.read()
        if not success:
            break

        resized = cv2.resize(frame, (960, 540))
        blurred = cv2.GaussianBlur(resized, (5, 5), 0)
        mask = subtractor.apply(blurred)
        mask = cv2.medianBlur(mask, 5)
        _, mask = cv2.threshold(mask, 200, 255, cv2.THRESH_BINARY)

        kernel = np.ones((5, 5), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=2)
        mask = cv2.dilate(mask, kernel, iterations=2)

        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        motion_boxes = []
        for contour in contours:
            if cv2.contourArea(contour) < min_area:
                continue
            x, y, w, h = cv2.boundingRect(contour)
            motion_boxes.append((x, y, w, h))

        results.append({
            "frame_index": frame_count,
            "frame": resized,
            "mask": mask,
            "boxes": motion_boxes,
        })
        frame_count += 1

    capture.release()
    return results


print("Motion detection helper ready.")

In [ ]:
def draw_motion_overlay(result):
    frame = result["frame"].copy()
    for x, y, w, h in result["boxes"]:
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
    return frame


def show_result(result):
    overlay = cv2.cvtColor(draw_motion_overlay(result), cv2.COLOR_BGR2RGB)
    mask = result["mask"]

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(overlay)
    axes[0].set_title(f"Frame {result['frame_index']} with motion boxes")
    axes[0].axis("off")

    axes[1].imshow(mask, cmap="gray")
    axes[1].set_title("Motion mask")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()


print("Visualization helpers ready.")

In [ ]:
VIDEO_PATH = ROOT / "data" / "samples" / "sample.mp4"

if VIDEO_PATH.exists():
    results = detect_motion_frames(VIDEO_PATH)
    print(f"Loaded {len(results)} frames from {VIDEO_PATH}")

    if results:
        print(f"Frame 0 has {len(results[0]['boxes'])} motion boxes")
        show_result(results[0])
else:
    print(f"Put a sample video at: {VIDEO_PATH}")
    print("Then rerun this cell to see motion detection output.")